# ⊙ Advanced RAG Masterclass · Practical Notebook

Run each cell in order and read the code comments.
At the end of each phase there are reflection questions — there is no single correct answer, they are designed to make you think.

**Estimated time: 25–30 minutes**

| Phase | Technique | What you will see |
|-------|-----------|-------------------|
| 01 | Hybrid Search | Why Dense alone fails on proper nouns |
| 02 | Reranking | Why order matters as much as finding the doc |
| 03 | CRAG | Why a correct doc can still be insufficient |

---
## 0 · Installation

We install two external libraries that do not ship with Python:

- **`sentence-transformers`** — loads pre-trained language models that convert text into numerical vectors (embeddings). Used for Dense Retrieval and the Cross-Encoder Reranker.
- **`rank_bm25`** — implements the BM25 algorithm, the keyword-matching search standard. No neural networks: it simply counts and weights term occurrences.

Run this cell once and wait ~2 minutes.

In [ ]:
!pip install -q sentence-transformers rank_bm25

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

print("✓ Dependencies ready")

---
## 1 · The Knowledge Base

These are the documents our RAG system has available.
Notice that for each test query there is **one correct doc** and **one trap doc**
that sounds related but would produce an incomplete or incorrect answer.

In [ ]:
# Each document simulates a "chunk" from a real knowledge base
# In production these would be fragments of PDFs, wikis, manuals, etc.
DOCS = [
    {"id": "d1", "title": "Scarlet Lynel: anatomy and attack patterns",
     "text": "The Scarlet Lynel is one of the strongest enemies in Hyrule. It has three phases: charge, sword combo and fireball. Its weak point is the head during its leap."},

    {"id": "d2", "title": "How to defeat the Scarlet Lynel: tactical guide",  # CORRECT for Q1
     "text": "To beat the Scarlet Lynel: execute a perfect dodge just before its charge to trigger slow-motion, shoot arrows at its head and then mount its back. Never block its sword combo — always dodge sideways."},

    {"id": "d3", "title": "Advanced close-range combat techniques",            # TRAP for Q1
     "text": "Advanced combat techniques include perfect shield parry, timed lateral dodge and immediate counter-attack. They are effective against high-difficulty bosses and minimise damage taken."},

    {"id": "d4", "title": "Zora's Water Temple: access and location",          # CORRECT for Q2
     "text": "Zora's Water Temple is accessed from Zora's Domain, in north-eastern Hyrule. The Zora Scale is required to enter. Coordinates: X=2365, Y=0412. The inner boss is Morpha."},

    {"id": "d5", "title": "Guide to Hyrule's aquatic kingdoms and zones",      # TRAP for Q2
     "text": "Hyrule's aquatic zones include Lake Hylia, Zora's Domain and Zora's Fountain. To explore them you need swimming gear or the Zora ability."},

    {"id": "d6", "title": "Fireproof Elixir: recipe and properties",           # CORRECT for Q3
     "text": "The Fireproof Elixir is crafted with Fire Salamander Dust x3 plus any insect body. Grants fire resistance level 2 for 8 minutes. No cooking pot needed — prepared in a field mortar."},

    {"id": "d7", "title": "Hyrule field alchemy and recipe manual",            # TRAP for Q3
     "text": "Field recipes allow you to craft elixirs without fire using natural ingredients. Combine insects with lizards or monster parts to create elemental resistance effects."},

    {"id": "d8", "title": "Goron Warrior Fireproof Armor",
     "text": "The Goron Warrior Fireproof Armor grants total fire immunity. Purchased at the Goron City Forge for 2400 rupees and cannot be crafted."},
]

# The 3 queries we will use to evaluate the system
QUERIES = [
    {"query": "How do I defeat the Scarlet Lynel?",      "correct": "d2", "trap": "d3"},
    {"query": "How do I access Zora's Water Temple?",    "correct": "d4", "trap": "d5"},
    {"query": "How do I craft the Fireproof Elixir?",    "correct": "d6", "trap": "d7"},
]

print(f"Knowledge base : {len(DOCS)} documents")
print(f"Test queries   : {len(QUERIES)}")

---
## 2 · The Problem with Basic RAG

### What is an embedding?

An **embedding** is the translation of a text into a list of numbers (a vector).
The model learns to place texts with similar meanings close together in that numerical space:

```
"dog"    →  [0.21, -0.54, 0.88, ...]   ─┐
"canine" →  [0.19, -0.51, 0.85, ...]   ─┴─ close (semantically similar)

"recipe" →  [-0.73, 0.12, -0.44, ...]  ─── far from "dog"
```

**Cosine similarity** measures the angle between two vectors: a small angle (vectors pointing in the same direction) means the texts are similar. With normalised vectors this reduces to a simple dot product.

### The blind spot

This approach works well for natural language but fails on specific terms:
the model does not know that **"Scarlet Lynel"** is a unique proper noun —
it treats it as a generic "beast" and may return documents about combat in general
when you need information about that specific enemy.

Observe what happens:

In [ ]:
# Load the embedding model
# This model converts any text into a vector of 384 numbers
# representing its "position" in a semantic space
print("Loading embedding model... (~1 min)")
model = SentenceTransformer("all-MiniLM-L6-v2")

# Convert all documents to vectors at once (batch)
# We concatenate title + text so the vector captures both
texts = [f"{d['title']}. {d['text']}" for d in DOCS]
doc_vectors = model.encode(texts, normalize_embeddings=True)
# normalize_embeddings=True → each vector has length 1
# this makes dot product equivalent to cosine similarity

print(f"✓ {len(DOCS)} documents converted to {doc_vectors.shape[1]}-dimensional vectors")

In [ ]:
def search_dense(query, top_k=4):
    # Convert the query into the same vector space as the docs
    query_vector = model.encode([query], normalize_embeddings=True)

    # Cosine similarity: how "similar" is the query vector to each doc
    # With normalised vectors, dot product = cosine similarity
    scores = (query_vector @ doc_vectors.T)[0]

    # Sort descending and take top_k
    indices = np.argsort(scores)[::-1][:top_k]
    return [(DOCS[i], float(scores[i])) for i in indices]


# ── Test with the first query ──────────────────────────────────
query = QUERIES[0]["query"]
print(f"Query: '{query}'")
print("-" * 65)

for i, (doc, score) in enumerate(search_dense(query), 1):
    mark = ""
    if doc["id"] == QUERIES[0]["correct"]: mark = "  ← CORRECT"
    if doc["id"] == QUERIES[0]["trap"]:    mark = "  ← TRAP ⚠"
    print(f"  #{i}  [{score:.3f}]  {doc['title']}{mark}")

### ⊙ Reflect

> **1.** The model probably placed "Advanced close-range combat techniques" high in the ranking.
> Why does this happen when that document does not mention the Scarlet Lynel specifically?

> **2.** "Scarlet Lynel", "Zora's Water Temple" and "Fireproof Elixir" are proper nouns.
> What specific difficulty do proper nouns pose for a semantic model?

> **3.** In what type of company knowledge base would this problem be most severe:
> an HR wiki or technical documentation with product codes?

---
## 3 · Phase 01: Hybrid Search

The solution is to combine two complementary search engines:

- **Dense** → understands meaning. Good for natural language and paraphrases.
- **BM25** → counts exact word matches. Good for technical terms and proper nouns.

Then we merge them with **RRF**: a document that ranks well in *both* lists wins.

In [ ]:
# BM25 works on tokens (words), not vectors
# So we simply tokenise by splitting on spaces
tokenised_corpus = [f"{d['title']} {d['text']}".lower().split() for d in DOCS]
bm25 = BM25Okapi(tokenised_corpus)

def search_bm25(query, top_k=4):
    # BM25 scores each doc based on how many query words it contains
    # and how often they appear relative to the document length
    scores = bm25.get_scores(query.lower().split())
    indices = np.argsort(scores)[::-1][:top_k]
    return [(DOCS[i], float(scores[i])) for i in indices]


# ── Compare Dense vs BM25 on the same query ───────────────────
query = QUERIES[0]["query"]
res_dense = search_dense(query, top_k=4)
res_bm25  = search_bm25(query, top_k=4)

print(f"Query: '{query}'\n")
print(f"{'#':<3}  {'DENSE (semantic)':<40}  BM25 (exact)")
print("-" * 80)
for i in range(4):
    d = res_dense[i][0]["title"][:37]
    b = res_bm25[i][0]["title"][:37]
    print(f"#{i+1:<2}  {d:<40}  {b}")

print("\n⊙ Note: BM25 detects 'Scarlet Lynel' as an exact term → promotes the correct doc")

In [ ]:
def merge_rrf(*rankings, k=60):
    # RRF assigns each doc a score based on its position in each list
    # Score = Σ 1 / (k + position)
    # k=60 is the standard value: smooths the gap between position #1 and #2
    scores = {}
    for ranking in rankings:
        for position, (doc, _) in enumerate(ranking, 1):
            doc_id = doc["id"]
            # Doc in position 1 contributes: 1/(60+1) = 0.016
            # Doc in position 5 contributes: 1/(60+5) = 0.015
            # The difference is small → prevents one engine from dominating
            scores[doc_id] = scores.get(doc_id, 0) + 1.0 / (k + position)

    sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    doc_by_id = {d["id"]: d for d in DOCS}
    return [(doc_by_id[doc_id], score) for doc_id, score in sorted_docs]


hybrid_result = merge_rrf(res_dense, res_bm25)

print(f"HYBRID Result (Dense + BM25 + RRF)\n")
print(f"{'#':<3}  {'Document':<45}  RRF Score")
print("-" * 65)
for i, (doc, score) in enumerate(hybrid_result[:4], 1):
    mark = ""
    if doc["id"] == QUERIES[0]["correct"]: mark = "  ← CORRECT"
    if doc["id"] == QUERIES[0]["trap"]:    mark = "  ← TRAP"
    print(f"#{i:<2}  {doc['title']:<45}  {score:.5f}{mark}")

### ⊙ Experiment

Change the value of `k` in the cell below and re-run.
Does the order change? Does the ranking become more or less stable?

In [ ]:
# ══ EXPERIMENT ZONE ══════════════════════════════════════════
K_RRF = 60   # try 5 (position matters a lot) or 200 (position matters little)
# ═════════════════════════════════════════════════════════════

result = merge_rrf(res_dense, res_bm25, k=K_RRF)

print(f"Ranking with k={K_RRF}:")
for i, (doc, score) in enumerate(result[:4], 1):
    mark = " ✓" if doc["id"] == QUERIES[0]["correct"] else " ✗" if doc["id"] == QUERIES[0]["trap"] else ""
    print(f"  #{i}  {doc['title']}{mark}")

In [ ]:
# ── Metric: Hit Rate @3 ───────────────────────────────────────
# Hit Rate = fraction of queries where the correct doc appears in the top-3
# If 0%, the system doesn't even find the relevant document

def hit_rate(queries, search_fn, k=3):
    hits = sum(
        q["correct"] in [doc["id"] for doc, _ in search_fn(q["query"])[:k]]
        for q in queries
    )
    return hits / len(queries)

def hybrid(query, k=5):
    return merge_rrf(search_dense(query, k), search_bm25(query, k))[:k]

hr_basic   = hit_rate(QUERIES, lambda q: search_dense(q, 3))
hr_hybrid  = hit_rate(QUERIES, lambda q: hybrid(q, 3))

print(f"Hit Rate @3  —  Dense only: {hr_basic:.0%}   Hybrid: {hr_hybrid:.0%}")
print(f"Improvement: +{(hr_hybrid - hr_basic)*100:.0f} percentage points")

### ⊙ Reflect

> **1.** RRF benefits documents that rank well in **both** lists.
> Why is that criterion more robust than trusting a single engine?

> **2.** What happened when you changed k?
> When would you prefer a small k vs a large k?

> **3.** In your company, what kind of terms do you think BM25 would rescue better than Dense?
> Think about acronyms, client names, internal product codes...

---
## 4 · Phase 02: Reranking

Finding the correct document in the top-3 is not enough if it sits at position #3.
LLMs suffer from the **"Lost in the Middle"** effect: they pay more attention to the beginning
and end of the context. A document at position #3 may be ignored.

The **Cross-Encoder** reads the query and the document *together* in a single pass,
allowing it to understand relevance with far greater precision than Dense Retrieval.

In [ ]:
print("Loading Cross-Encoder... (~1 min)")
# This model was specifically trained to assign a relevance score
# to (query, document) pairs. Slower but much more accurate.
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("✓ Cross-Encoder ready")

In [ ]:
def rerank(query, candidates, top_n=3):
    # Build (query, doc_text) pairs for each candidate
    pairs = [(query, f"{doc['title']}. {doc['text']}") for doc, _ in candidates]

    # The Cross-Encoder evaluates all pairs and returns a score per pair
    # Unlike Dense, query and doc are processed TOGETHER here
    scores = cross_encoder.predict(pairs)

    # Pair each doc with its new score and sort
    with_score = [(candidates[i][0], float(scores[i])) for i in range(len(candidates))]
    return sorted(with_score, key=lambda x: x[1], reverse=True)[:top_n]


# ── Compare before and after reranking ────────────────────────
query = QUERIES[0]["query"]
candidates = hybrid(query, k=5)
reranked = rerank(query, candidates)

print("BEFORE reranking (Hybrid top-4):")
for i, (doc, _) in enumerate(candidates[:4], 1):
    mark = "  ← CORRECT" if doc["id"] == QUERIES[0]["correct"] else ""
    mark = "  ← TRAP"    if doc["id"] == QUERIES[0]["trap"]    else mark
    print(f"  #{i}  {doc['title']}{mark}")

print("\nAFTER reranking (Cross-Encoder top-3):")
for i, (doc, score) in enumerate(reranked, 1):
    mark = "  ← CORRECT" if doc["id"] == QUERIES[0]["correct"] else ""
    mark = "  ← TRAP"    if doc["id"] == QUERIES[0]["trap"]    else mark
    print(f"  #{i}  [{score:+.2f}]  {doc['title']}{mark}")

### ⊙ Experiment

Change how many candidates you pass to the Cross-Encoder.
Is there a difference between passing 3 vs 7?

In [ ]:
# ══ EXPERIMENT ZONE ══════════════════════════════════════════
CANDIDATES_FOR_RERANKER = 5   # try 3 or 7
DOCS_TO_LLM             = 3   # how many pass to the LLM after reranking
# ═════════════════════════════════════════════════════════════

for q in QUERIES:
    cands = hybrid(q["query"], k=CANDIDATES_FOR_RERANKER)
    top   = rerank(q["query"], cands, top_n=DOCS_TO_LLM)
    top1  = top[0][0]
    mark  = "✓ CORRECT" if top1["id"] == q["correct"] else "✗ WRONG"
    print(f"  {mark}  |  {q['query'][:45]:<45}  →  {top1['title'][:40]}")

In [ ]:
# ── Metric: MRR (Mean Reciprocal Rank) ────────────────────────
# MRR measures the exact position of the correct doc
# Position 1 → 1.0 (perfect), position 2 → 0.5, position 3 → 0.33

def pipeline_hybrid_rerank(query, k=5):
    return rerank(query, hybrid(query, k), top_n=k)

def mrr(queries, search_fn, k=5):
    total = 0.0
    for q in queries:
        for pos, (doc, _) in enumerate(search_fn(q["query"])[:k], 1):
            if doc["id"] == q["correct"]:
                total += 1.0 / pos
                break
    return total / len(queries)

mrr_basic    = mrr(QUERIES, lambda q: search_dense(q, 5))
mrr_hybrid   = mrr(QUERIES, lambda q: hybrid(q, 5))
mrr_reranked = mrr(QUERIES, pipeline_hybrid_rerank)

print(f"MRR  —  Dense: {mrr_basic:.2f}   Hybrid: {mrr_hybrid:.2f}   + Reranking: {mrr_reranked:.2f}")
print(f"Total improvement vs basic: +{(mrr_reranked - mrr_basic)*100:.1f} points")

### ⊙ Reflect

> **1.** The Cross-Encoder is slower because it processes query + document together.
> Dense Retrieval processes them separately. Why does that difference make the Cross-Encoder more accurate?

> **2.** If your system receives 1,000 queries per hour, would you apply reranking to all of them?
> For what type of queries would it be highest priority?

> **3.** What is the practical difference for the end user between an MRR of 0.5 and an MRR of 1.0
> when they are waiting for an answer?

---
## 5 · Phase 03: CRAG (Corrective RAG)

Even with the best document at #1, it may happen that the document is
**outdated**, **partially incorrect** or simply **does not cover the question**.

In basic RAG, the LLM would answer anyway, inventing the missing part: a **hallucination**.

CRAG adds an **LLM Judge** that evaluates the document before passing it to the generator LLM:

| Score | Verdict | Action |
|-------|---------|--------|
| ≥ 0.80 | CORRECT | Pass directly to LLM |
| 0.40 – 0.79 | AMBIGUOUS | Internal doc + web search |
| < 0.40 | INCORRECT | Web search only |

In [ ]:
# In production the LLM Judge is a real API call to Claude Haiku or GPT-4-mini
# Here we use pre-calculated scores to avoid needing an API key
# The behaviour is identical: score = how well the doc answers the query
JUDGE_SCORES = {
    ("How do I defeat the Scarlet Lynel?",    "d1"): 0.55,  # anatomy → missing full tactics
    ("How do I defeat the Scarlet Lynel?",    "d2"): 0.94,  # complete tactical guide ✓
    ("How do I defeat the Scarlet Lynel?",    "d3"): 0.48,  # generic combat → imprecise
    ("How do I access Zora's Water Temple?",  "d4"): 0.67,  # location ok, access reqs possibly outdated
    ("How do I access Zora's Water Temple?",  "d5"): 0.31,  # generic → no specific access info
    ("How do I craft the Fireproof Elixir?",  "d6"): 0.91,  # exact recipe ✓
    ("How do I craft the Fireproof Elixir?",  "d7"): 0.43,  # generic → no exact ingredients
}

def evaluate_crag(query, document, threshold_correct=0.80, threshold_ambiguous=0.40):
    score = JUDGE_SCORES.get((query, document["id"]), 0.5)

    if score >= threshold_correct:
        return score, "CORRECT",    "→ pass directly to LLM"
    elif score >= threshold_ambiguous:
        return score, "AMBIGUOUS",  "→ internal doc + web search"
    else:
        return score, "INCORRECT",  "→ web search only"


# ── Evaluate all 3 queries ────────────────────────────────────
print("CRAG evaluates the top-1 document from the pipeline:\n")

for q in QUERIES:
    cands = hybrid(q["query"], k=5)
    top1, _ = rerank(q["query"], cands, top_n=1)[0]
    score, verdict, action = evaluate_crag(q["query"], top1)
    icon = {"CORRECT": "✓", "AMBIGUOUS": "◎", "INCORRECT": "✗"}[verdict]
    print(f"  {icon} [{score:.2f}]  {q['query']}")
    print(f"         Top-1 doc: {top1['title']}")
    print(f"         {action}\n")

### ⊙ Experiment

Raise the CORRECT threshold to `0.98`. What happens to all 3 queries?
What is the implication for cost and speed?

In [ ]:
# ══ EXPERIMENT ZONE ══════════════════════════════════════════
THRESHOLD_CORRECT   = 0.80   # try 0.98 — how many trigger web search?
THRESHOLD_AMBIGUOUS = 0.40   # try 0.60 — does any verdict change?
# ═════════════════════════════════════════════════════════════

no_web = 0
with_web = 0

for q in QUERIES:
    cands = hybrid(q["query"], k=5)
    top1, _ = rerank(q["query"], cands, top_n=1)[0]
    score, verdict, action = evaluate_crag(q["query"], top1, THRESHOLD_CORRECT, THRESHOLD_AMBIGUOUS)
    icon = {"CORRECT": "✓", "AMBIGUOUS": "◎", "INCORRECT": "✗"}[verdict]

    if verdict == "CORRECT": no_web += 1
    else: with_web += 1

    print(f"  {icon} [{score:.2f}]  {q['query'][:45]:<45}  {action}")

print(f"\n  No web search: {no_web}/3   With web search: {with_web}/3")

### ⊙ Reflect

> **1.** A very high threshold (0.98) almost always triggers web search.
> A very low threshold (0.20) almost never triggers it.
> What would be the right criterion to choose the threshold in a real system?

> **2.** In what industry or application do you think the cost of a hallucination justifies
> always running full CRAG? In which one does it not?

> **3.** The LLM Judge consumes tokens (= money) on every evaluation.
> How would you optimise this if you have 10,000 queries per day?

---
## 6 · Final Comparison

We measure the cumulative impact of the three phases on the same 3 queries.

**How to read the table:**
- **Hit Rate @3** — Does the correct doc appear anywhere in the top-3? The basic system may have it but buried; Hybrid Search ensures it is always within the top-3.
- **MRR** — At exactly what position? An MRR of 1.0 means the correct doc is always #1. Reranking is responsible for moving it to first place.
- **Faithfulness** — Is the doc that finally reaches the LLM the correct one? CRAG closes the pipeline by discarding or complementing any doc that is still insufficient.

A system with high Hit Rate but low MRR has the documents but in the wrong order.
A system with high MRR but low Faithfulness found and sorted well, but the LLM can still hallucinate.

In [ ]:
def full_pipeline(query, k=5):
    # Phase 01: Hybrid
    candidates = merge_rrf(search_dense(query, k), search_bm25(query, k))[:k]
    # Phase 02: Reranking
    return rerank(query, candidates, top_n=k)

# ── Hit Rate @3 ───────────────────────────────────────────────
hr_basic    = hit_rate(QUERIES, lambda q: search_dense(q, 3))
hr_advanced = hit_rate(QUERIES, lambda q: full_pipeline(q, 5)[:3])

# ── MRR ──────────────────────────────────────────────────────
mrr_basic    = mrr(QUERIES, lambda q: search_dense(q, 5))
mrr_advanced = mrr(QUERIES, full_pipeline)

# ── Faithfulness (proxy) ──────────────────────────────────────
# Does the doc that reaches the LLM match the correct one?
faith = sum(
    full_pipeline(q["query"])[0][0]["id"] == q["correct"]
    for q in QUERIES
) / len(QUERIES)

print("╔══════════════════════════════════════════════════════╗")
print("║           FINAL COMPARISON · 3 QUERIES              ║")
print("╠══════════════════╦═════════╦═══════════╦════════════╣")
print("║  Metric          ║  Basic  ║ Pipeline  ║  Gain      ║")
print("╠══════════════════╬═════════╬═══════════╬════════════╣")
print(f"║  Hit Rate @3     ║  {hr_basic:>5.0%}  ║   {hr_advanced:>5.0%}   ║  +{(hr_advanced-hr_basic)*100:.0f} pp       ║")
print(f"║  MRR             ║  {mrr_basic:>5.2f}  ║   {mrr_advanced:>5.2f}   ║  +{(mrr_advanced-mrr_basic)*100:.1f} pp      ║")
print(f"║  Faithfulness    ║    N/A  ║   {faith:>5.0%}   ║   —          ║")
print("╚══════════════════╩═════════╩═══════════╩════════════╝")
print()
print("  Hit Rate: Phase 01 (Hybrid Search)  ensures the doc is in the top-3")
print("  MRR:      Phase 02 (Reranking)      moves it to position #1")
print("  Faith.:   Phase 03 (CRAG)           verifies what reaches the LLM is correct")

---
## 7 · Challenge

You have all the pipeline functions available.
Your goal is to implement `analyse_gain(queries)` which answers this question:

**Which query benefited most from the full pipeline vs Dense alone, and in which phase did the biggest improvement occur?**

The function must:
1. For each query, calculate MRR with Dense only, with Hybrid only, and with the full pipeline
2. Calculate the total MRR gain per query (`MRR_pipeline − MRR_dense`)
3. Identify the query with the highest gain
4. Determine whether that gain came mainly from Phase 01 (Hybrid) or Phase 02 (Reranking)

Available functions: `search_dense()`, `hybrid()`, `rerank()`, `full_pipeline()`

**Expected output** (numbers will vary by model, but the format must match):
```
Query                                          Dense  Hybrid  Pipeline    Gain
-------------------------------------------------------------------------------------
How do I defeat the Scarlet Lynel?             0.33    1.00      1.00   +0.67  ← Phase 01 (Hybrid)
How do I access Zora's Water Temple?           0.33    1.00      1.00   +0.67  ← Phase 01 (Hybrid)
How do I craft the Fireproof Elixir?           0.50    1.00      1.00   +0.50  ← Phase 01 (Hybrid)

⊙ Highest gain: 'How do I defeat the Scarlet Lynel?' (+0.67 MRR)
```

In [ ]:
# ══ CHALLENGE ════════════════════════════════════════════════

def position_of_correct_doc(results, correct_id, k=5):
    """Returns the 1-based position of the correct doc in the results list.
    If not found in the first k, returns k+1 (not found)."""
    # Your code here
    pass


def mrr_single(results, correct_id):
    """Calculates MRR for a single query given its result list."""
    # Your code here
    pass


def analyse_gain(queries):
    """For each query calculates MRR at three pipeline checkpoints and compares."""
    print(f"{'Query':<45}  {'Dense':>6}  {'Hybrid':>7}  {'Pipeline':>9}  {'Gain':>6}")
    print("-" * 85)

    best_query = None
    best_gain  = -1

    for q in queries:
        # Step 1: Dense only
        res_dense    = search_dense(q["query"], top_k=5)

        # Step 2: Hybrid only (no reranking)
        res_hybrid   = hybrid(q["query"], k=5)

        # Step 3: Full pipeline (Hybrid + Reranking)
        res_pipeline = full_pipeline(q["query"], k=5)

        # Calculate the three MRRs
        mrr_d = mrr_single(res_dense,    q["correct"])
        mrr_h = mrr_single(res_hybrid,   q["correct"])
        mrr_p = mrr_single(res_pipeline, q["correct"])

        gain = mrr_p - mrr_d

        # Determine which phase drove the biggest improvement
        if (mrr_h - mrr_d) >= (mrr_p - mrr_h):
            key_phase = "Phase 01 (Hybrid)"
        else:
            key_phase = "Phase 02 (Reranking)"

        print(f"{q['query'][:44]:<45}  {mrr_d:>6.2f}  {mrr_h:>7.2f}  {mrr_p:>9.2f}  {gain:>+6.2f}  ← {key_phase}")

        if gain > best_gain:
            best_gain  = gain
            best_query = q["query"]

    print(f"\n⊙ Highest gain: '{best_query}' (+{best_gain:.2f} MRR)")


analyse_gain(QUERIES)

---
## 8 · Summary and Closing

```
User query
     │
     ▼  PHASE 01 · Hybrid Search  →  KPI: Hit Rate
     Dense + BM25  →  RRF  →  Top-K candidates
     │   (never misses exact terms or natural language)
     │
     ▼  PHASE 02 · Reranking  →  KPI: MRR
     Cross-Encoder  →  Top-3 reordered
     │   (correct doc always at position #1)
     │
     ▼  PHASE 03 · CRAG  →  KPI: Faithfulness
     LLM Judge  →  Correct / Ambiguous / Incorrect
     │   (if it fails, searches the web before answering)
     │
     ▼  Generator LLM with verified context
     Hallucination-free answer
```

### Taking it to production
- **LLM Judge**: use Claude Haiku or GPT-4o-mini (cheaper for evaluation)
- **Spanish embeddings**: `intfloat/multilingual-e5-large` outperforms MiniLM in Spanish
- **Web search**: Tavily Search API integrates in one line with LangChain
- **Real Faithfulness evaluation**: frameworks like RAGAS or deepeval

### ⊙ Closing question

> Think of a RAG system you know or one you would like to build.
> Which of the three phases do you think would have the biggest impact in that specific case?
> Why?